# PosEnc Output Comparison

This notebook loads experiment artifacts (`vectors.npy`, `metadata.json`, `encoded_*.npy`) and compares encoder behavior.

In [ ]:
from pathlib import Path
import json
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print('matplotlib unavailable:', exc)
    print('Install notebook deps with: uv sync --extra notebooks')

In [ ]:
CANDIDATE_RUN_DIRS = [
    Path('out'),
    Path('../out'),
    Path('/home/jake/Developer/posenc/out'),
]

RUN_DIR = next((p for p in CANDIDATE_RUN_DIRS if p.exists()), CANDIDATE_RUN_DIRS[0])
print('RUN_DIR:', RUN_DIR.resolve())

def load_artifacts(run_dir: Path):
    meta_path = run_dir / 'metadata.json'
    vectors_path = run_dir / 'vectors.npy'
    if not meta_path.exists() or not vectors_path.exists():
        raise FileNotFoundError(
            f'Missing artifacts in {run_dir}. Expected metadata.json and vectors.npy.'
        )

    metadata = json.loads(meta_path.read_text())
    vectors = np.load(vectors_path)
    encoded = {
        p.stem.replace('encoded_', ''): np.load(p)
        for p in sorted(run_dir.glob('encoded_*.npy'))
    }
    return metadata, vectors, encoded

try:
    metadata, vectors, encoded = load_artifacts(RUN_DIR)
    print('encoders with saved tensors:', list(encoded.keys()))
    print('vectors shape:', vectors.shape)
except FileNotFoundError as exc:
    print(exc)
    print('Generate artifacts with:')
    print('uv run python main.py --encoders all --save-dir out --save-encoded')
    metadata = None
    vectors = None
    encoded = {}

In [ ]:
if not encoded:
    print('No encoded tensors loaded; skipping comparison plot.')
else:
    mean_norm_by_encoder = {}
    for name, tensor in encoded.items():
        norms = np.linalg.norm(tensor, axis=2)
        mean_norm_by_encoder[name] = float(np.mean(norms))

    if HAVE_MPL:
        plt.figure(figsize=(8, 4))
        plt.bar(mean_norm_by_encoder.keys(), mean_norm_by_encoder.values())
        plt.ylabel('mean output norm')
        plt.title('Mean encoded norm by encoder')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('Mean encoded norm by encoder:')
        for name, value in mean_norm_by_encoder.items():
            print(f'  {name}: {value:.6f}')